# 03 — Evaluate

This notebook evaluates trained model predictions against the **test set**
(the last portion of the date range defined in `config.yaml`).

It reads the inference-output parquets produced by `04_inference.ipynb`
(run it first on your test dates) and computes error metrics per station,
per forecast hour, and per met variable.

**Steps**
1. Load config and locate inference-output parquets.
2. Compute per-station MAE, RMSE, and bias.
3. Plot a station-level summary table and choropleth map.
4. Plot bulk scatter density and time-of-day heatmaps.

**Prerequisite**: run `04_inference.ipynb` over the test period first,
or run inference on your own via the CLI.

In [ ]:
import sys, os

REPO_ROOT = os.path.abspath("../..")
APP_ROOT  = os.path.abspath("..")
for p in (REPO_ROOT, APP_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(APP_ROOT)
print("Working directory:", os.getcwd())

In [ ]:
from app.utils.config_loader import load_config

cfg = load_config()
print("Results directory:", cfg.output.results_dir)
print("Model type       :", cfg.model)
print("Forecast hours   :", cfg.data.forecast_hours)
print("Met variables    :", cfg.data.metvars)

## Step 1 — Load inference output parquets

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

results_dir = Path(cfg.output.results_dir)

all_preds = []
for fh in cfg.data.forecast_hours:
    for metvar in cfg.data.metvars:
        pq_path = results_dir / f"fh{fh}_{metvar}_inference_out.parquet"
        if pq_path.exists():
            df = pd.read_parquet(pq_path).reset_index()
            df["fh"] = fh
            df["metvar"] = metvar
            all_preds.append(df)
        else:
            print(f"[MISSING] {pq_path.name} — run 04_inference.ipynb first.")

if not all_preds:
    raise RuntimeError(
        "No inference-output parquets found.  Run 04_inference.ipynb over "
        "the test period first."
    )

preds = pd.concat(all_preds, ignore_index=True)
print(f"Loaded {len(preds):,} prediction rows across {preds['fh'].nunique()} fh "
      f"and {preds['metvar'].nunique()} metvars.")
preds.head()

## Step 2 — Per-station error metrics

In [ ]:
# The output column holding the model prediction is 'model_output'.
# The column holding the realised error is 'actual' (if available from
# eval_model_metrics) or we derive it on the fly below.
pred_col = "model_output"
target_col = "actual"  # produced by eval_model_metrics.py

if target_col not in preds.columns:
    print(
        f"Column '{target_col}' not present — loading error_metrics parquets to join."
    )
    metric_frames = []
    for fh in cfg.data.forecast_hours:
        for metvar in cfg.data.metvars:
            mp = results_dir / f"fh{fh}_{metvar}_error_metrics.parquet"
            if mp.exists():
                mf = pd.read_parquet(mp).reset_index()
                mf["fh"] = fh
                mf["metvar"] = metvar
                metric_frames.append(mf)
    if metric_frames:
        metrics = pd.concat(metric_frames, ignore_index=True)
        preds = preds.merge(
            metrics[["valid_time", "stid", "fh", "metvar", "actual"]],
            on=["valid_time", "stid", "fh", "metvar"],
            how="left",
        )

# Clip to the test period only.
preds["valid_time"] = pd.to_datetime(preds["valid_time"])
test_start = pd.Timestamp(cfg.training.val_end)
test_end   = pd.Timestamp(cfg.training.test_end)
preds_test = preds[(preds["valid_time"] >= test_start) & (preds["valid_time"] <= test_end)]
print(f"Test period rows: {len(preds_test):,}  ({test_start.date()} → {test_end.date()})")

In [ ]:
# Compute MAE, RMSE, Bias per (station, fh, metvar)
if "actual" in preds_test.columns:
    station_col = "stid" if "stid" in preds_test.columns else "station"
    valid = preds_test.dropna(subset=[pred_col, "actual"])
    valid["error"] = valid[pred_col] - valid["actual"]

    summary = (
        valid.groupby([station_col, "fh", "metvar"])
        .agg(
            MAE=("error", lambda x: np.abs(x).mean()),
            RMSE=("error", lambda x: np.sqrt((x**2).mean())),
            Bias=("error", "mean"),
            n=("error", "count"),
        )
        .reset_index()
    )
    print(summary.groupby(["fh", "metvar"])[["MAE", "RMSE", "Bias"]].mean().round(4))
else:
    print("'actual' column not available — skipping metric computation.")
    summary = pd.DataFrame()

In [ ]:
# Interactive table — sorted by worst MAE
if not summary.empty:
    display(summary.sort_values("MAE", ascending=False).head(30))

## Step 3 — Station choropleth map

Colour each station by its mean absolute error for the first forecast hour
in the config.

In [ ]:
import folium
import pandas as pd
from pathlib import Path

station_col = "stid" if "stid" in preds_test.columns else "station"

# Load station lat/lon from the metadata CSV
meta_path = Path(cfg.output.models_dir) / "lookups" / "station_meta.csv"
station_meta = pd.read_csv(meta_path) if meta_path.exists() else pd.DataFrame()

fh_plot   = cfg.data.forecast_hours[0]
var_plot  = cfg.data.metvars[0]

centre_lat = (cfg.bbox.lat_min + cfg.bbox.lat_max) / 2
centre_lon = (cfg.bbox.lon_min + cfg.bbox.lon_max) / 2
m = folium.Map(location=[centre_lat, centre_lon], zoom_start=7, tiles="CartoDB positron")

if not summary.empty and not station_meta.empty:
    sub = summary[(summary["fh"] == fh_plot) & (summary["metvar"] == var_plot)]
    sub = sub.merge(station_meta.rename(columns={"station": station_col}),
                    on=station_col, how="left")

    max_mae = sub["MAE"].max()
    for _, row in sub.dropna(subset=["lat", "lon", "MAE"]).iterrows():
        # Colour gradient: green (low MAE) → red (high MAE)
        frac = row["MAE"] / max_mae if max_mae > 0 else 0
        r = int(255 * frac)
        g = int(255 * (1 - frac))
        colour = f"#{r:02x}{g:02x}00"
        folium.CircleMarker(
            location=[row.lat, row.lon],
            radius=7,
            color=colour,
            fill=True,
            fill_color=colour,
            fill_opacity=0.85,
            tooltip=(
                f"{row[station_col]} | fh={fh_plot} {var_plot} | "
                f"MAE={row['MAE']:.4f} | Bias={row['Bias']:.4f}"
            ),
        ).add_to(m)
    print(f"Showing fh={fh_plot}, metvar={var_plot}")
else:
    print("No data to display — check that summary and station_meta are loaded.")

display(m)

## Step 4 — Bulk scatter density

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fh_plot  = cfg.data.forecast_hours[0]
var_plot = cfg.data.metvars[0]

if "actual" in preds_test.columns:
    sub = preds_test[
        (preds_test["fh"] == fh_plot) &
        (preds_test["metvar"] == var_plot)
    ].dropna(subset=[pred_col, "actual"])

    x = sub["actual"].values
    y = sub[pred_col].values

    fig, ax = plt.subplots(figsize=(6, 6))
    h = ax.hist2d(x, y, bins=60, cmap="plasma", norm=mcolors.LogNorm())
    plt.colorbar(h[3], ax=ax, label="Count (log)")

    lim = max(abs(x).max(), abs(y).max()) * 1.05
    ax.plot([-lim, lim], [-lim, lim], "w--", lw=1, label="1:1")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel("Actual forecast error")
    ax.set_ylabel("Predicted forecast error")
    ax.set_title(f"Bulk scatter — fh={fh_plot}  {var_plot}  (test set)")
    ax.legend()
    plt.tight_layout()
    scatter_path = Path(cfg.output.results_dir) / f"scatter_fh{fh_plot}_{var_plot}.png"
    plt.savefig(scatter_path, dpi=150)
    plt.show()
    print("Saved to", scatter_path)
else:
    print("'actual' column not available — skipping scatter plot.")

## Step 5 — Time-of-day error heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if "actual" in preds_test.columns:
    sub = preds_test.copy().dropna(subset=[pred_col, "actual"])
    sub["hour"] = sub["valid_time"].dt.hour
    sub["abs_error"] = (sub[pred_col] - sub["actual"]).abs()

    pivot = (
        sub.groupby(["metvar", "hour"])["abs_error"]
        .mean()
        .unstack("hour")
        .reindex(columns=range(24))
    )

    fig, ax = plt.subplots(figsize=(14, max(2, len(pivot) * 0.8)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
    plt.colorbar(im, ax=ax, label="Mean absolute error")
    ax.set_xticks(range(24))
    ax.set_xticklabels([f"{h:02d}Z" for h in range(24)], fontsize=8)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title("Mean absolute error by hour-of-day (UTC) and met variable")
    plt.tight_layout()
    tod_path = Path(cfg.output.results_dir) / "time_of_day_mae.png"
    plt.savefig(tod_path, dpi=150)
    plt.show()
    print("Saved to", tod_path)
else:
    print("'actual' column not available — skipping time-of-day heatmap.")

## Step 6 — BNN uncertainty plots (BNN model only)

In [ ]:
if cfg.model == "bnn":
    unc_cols = [c for c in preds_test.columns
                if c in ("epistemic_var", "aleatoric_var", "total_var")]
    if unc_cols:
        import matplotlib.pyplot as plt

        sub = preds_test[(preds_test["fh"] == cfg.data.forecast_hours[0]) &
                         (preds_test["metvar"] == cfg.data.metvars[0])].copy()

        for col in unc_cols:
            # Each cell in the col might be a list (one value per forecast step);
            # take the first step for a quick scalar plot.
            sub[f"{col}_s0"] = sub[col].apply(
                lambda v: v[0] if hasattr(v, '__getitem__') else v
            )

        fig, axes = plt.subplots(1, len(unc_cols), figsize=(5 * len(unc_cols), 4))
        if len(unc_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, unc_cols):
            ax.hist(sub[f"{col}_s0"].dropna(), bins=50, color="#457b9d", edgecolor="white")
            ax.set_xlabel(f"{col} (step 0)")
            ax.set_ylabel("Count")
            ax.set_title(col.replace("_", " ").title())
        plt.suptitle(
            f"BNN uncertainty — fh={cfg.data.forecast_hours[0]}  "
            f"{cfg.data.metvars[0]}  (test set)"
        )
        plt.tight_layout()
        unc_path = Path(cfg.output.results_dir) / "bnn_uncertainty_histograms.png"
        plt.savefig(unc_path, dpi=150)
        plt.show()
        print("Saved to", unc_path)
    else:
        print("No uncertainty columns found in predictions.")
else:
    print(f"BNN uncertainty plots are only available for model='bnn' (current: '{cfg.model}').")